# Stage 14: Regression & Prediction Enhancement
**Project**: Cyber Crime Analytics for National Security  
**Syllabus Alignment**: Unit 5 — Regression & Prediction (Linear Regression, Non-Linear Polynomial Regression, Regularized Regression: Ridge & Lasso, Tree-based & Ensemble Regression: Decision Tree, Random Forest, Gradient Boosting, Error Measurement & Residual Diagnostics: MAE, RMSE, $R^2$, Median AE, Model Complexity vs. Generalization)  
**Data Source**: Longitudinal State/UT Historical Panel (2018–2022 Rajya Sabha Cybercrime Series)  
**Sample Size**: $N = 106$ Observations ($N_{\text{train}} = 70$ for target years 2020 & 2021; $N_{\text{test}} = 36$ for held-out target year 2022)  

---

## 1. Objective & Analytical Scope

### Analytical Question:
> **"Can historical longitudinal volume patterns predict 1-year-ahead aggregate State/UT cybercrime case totals without contemporaneous sub-category identities or tautological features?"**

### Stage 7 Baseline Preservation & Extension Scope:
- **Stage 7 Validated Benchmark**: Stage 7 established that Log-Linear OLS achieves the lowest error and highest variance explained on the 2022 test horizon ($\text{MAE} = 479.37$, $\text{RMSE} = 1,143.46$, $R^2 = 0.9000$).
- **Stage 14 Analytical Expansion**: We evaluate whether adding **non-linear polynomial expansions** (Degree-2 and Degree-3 with Ridge regularizers), **non-linear regression trees**, and **ensemble methods** (Random Forest and Gradient Boosting) improves out-of-sample predictive accuracy over the simpler Log-Linear specification.
- **Strict Zero-Leakage Protocol**: Predictors are constructed strictly from years $t-1$ and $t-2$. Zero contemporaneous target-year features or 2023 sectional attributes are used.
- **Evaluation Metric Discipline**: All model predictions are transformed back to the **original case-count scale** via $\text{expm1}$ before computing MAE, RMSE, $R^2$, and Median Absolute Error.


In [1]:
# Setup environment, imports, and display configurations
import os
import sys
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

project_root = Path.cwd().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.regression_enhancement import (
    load_and_construct_regression_panel,
    train_and_evaluate_enhanced_regression,
    plot_regression_enhancement_figures,
    save_stage14_tables
)

print(f"Python Version: {sys.version.split()[0]}")
print(f"Working Directory: {project_root}")


Python Version: 3.13.7
Working Directory: C:\Users\kaval vyas\OneDrive\Desktop\Projects\cybercrime-analytics


## 2. Historical Longitudinal Panel Construction

The longitudinal dataset constructs records of $(State/UT, \text{target\_year})$ with lagged historical indicators:
- **Target Year 2020** (Train): Predictors from 2019 ($t-1$) and 2018 ($t-2$). $N = 35$ (Ladakh missing in 2018/19).
- **Target Year 2021** (Train): Predictors from 2020 ($t-1$) and 2019 ($t-2$). $N = 35$.
- **Target Year 2022** (Held-Out Test): Predictors from 2021 ($t-1$) and 2020 ($t-2$). $N = 36$ (Ladakh present in 2020/21).


In [2]:
# Load and inspect the longitudinal regression panel
panel_df = load_and_construct_regression_panel('data/processed/trend_2018_2022.csv')

print(f"Total Longitudinal Panel Size: {len(panel_df)} observations")
print(f"Training Partition (2020 & 2021): N = {(panel_df['split'] == 'train').sum()}")
print(f"Held-Out Test Partition (2022): N = {(panel_df['split'] == 'test').sum()}\n")

display(panel_df.head(10))


Total Longitudinal Panel Size: 106 observations
Training Partition (2020 & 2021): N = 70
Held-Out Test Partition (2022): N = 36



,state_name,target_year,split,lag_1,lag_2,lag_diff,lag_growth_rate,lag_mean_2yr,log_lag_1,log_lag_2,target_actual
0,Andaman and Nicobar Islands,2020,train,2.0,7.0,-5.0,-0.625000,4.5,1.098612,2.079442,5.0
1,Andhra Pradesh,2020,train,1886.0,1207.0,679.0,0.562086,1546.5,7.542744,7.096721,1899.0
2,Arunachal Pradesh,2020,train,8.0,7.0,1.0,0.125000,7.5,2.197225,2.079442,30.0
3,Assam,2020,train,2231.0,2022.0,209.0,0.103312,2126.5,7.710653,7.612337,3530.0
4,Bihar,2020,train,1050.0,374.0,676.0,1.802667,712.0,6.957497,5.926926,1512.0
5,Chandigarh,2020,train,23.0,30.0,-7.0,-0.225806,26.5,3.178054,3.433987,17.0
6,Chhattisgarh,2020,train,175.0,139.0,36.0,0.257143,157.0,5.170484,4.941642,297.0
7,Dadra and Nagar Haveli and Daman and Diu,2020,train,3.0,0.0,3.0,3.000000,1.5,1.386294,0.000000,3.0
8,Delhi,2020,train,115.0,189.0,-74.0,-0.389474,152.0,4.753590,5.247024,168.0
9,Goa,2020,train,15.0,29.0,-14.0,-0.466667,22.0,2.772589,3.401197,40.0


## 3. Model Evaluation Suite across 14 Architectures

We train 14 regression models on the training partition ($N_{\text{train}}=70$) and evaluate their performance on the held-out 2022 test partition ($N_{\text{test}}=36$).


In [3]:
# Train and evaluate all 14 regression architectures
comp_df, test_pred_df, err_df, res_summary_df, models_preds = train_and_evaluate_enhanced_regression(panel_df)

print("=== COMPREHENSIVE MODEL COMPARISON (HELD-OUT 2022 TEST HORIZON) ===")
display(comp_df)


=== COMPREHENSIVE MODEL COMPARISON (HELD-OUT 2022 TEST HORIZON) ===


,Model,Model_Category,Feature_Representation,Target_Transformation,MAE,RMSE,R2,Median_AE,Train_Obs,Test_Obs
0,Log-Linear OLS (Stage 7 Benchmark),Log-Linear OLS,"log_lag_1, log_lag_2",Log1p -> Expm1,479.37,1143.46,0.9000,69.85,70,36
1,Historical 2-Year Moving Average,Baseline,"lag_1, lag_2 Mean",None (Raw),563.04,1523.73,0.8224,57.00,70,36
2,Naive Persistent (Lag-1),Baseline,lag_1 (t-1),None (Raw),564.75,1340.75,0.8625,61.50,70,36
3,"Random Forest Regressor (Raw, depth=3)",Ensemble Forest,"lag_1, lag_2 (Raw)",None (Raw),576.80,1548.49,0.8166,75.69,70,36
4,Decision Tree Regressor (depth=3),Decision Tree,"lag_1, lag_2 (Raw)",None (Raw),608.42,1571.47,0.8111,100.86,70,36
5,"Log-Polynomial Degree 2 (Ridge, alpha=1.0)",Log-Polynomial Regularized,Degree-2 Log Features,Log1p -> Expm1,621.24,1465.82,0.8356,48.02,70,36
6,Log-Polynomial Degree 2 (OLS),Log-Polynomial,Degree-2 Log Features,Log1p -> Expm1,643.44,1520.52,0.8231,47.42,70,36
7,Random Forest Regressor (Log Target),Ensemble Forest,"log_lag_1, log_lag_2",Log1p -> Expm1,685.20,1735.34,0.7696,111.11,70,36
8,"Gradient Boosting (Raw, depth=2)",Ensemble Boosting,"lag_1, lag_2 (Raw)",None (Raw),728.33,1732.42,0.7704,197.17,70,36
9,"Ridge Regression (Raw, alpha=1.0)",Linear Regularized,"lag_1, lag_2 (Raw)",None (Raw),776.08,1680.13,0.7840,205.07,70,36


## 4. Test-Level Prediction & Error Analysis

We inspect state-level actuals, predictions, signed errors ($y - \hat{y}$), absolute errors ($|y - \hat{y}|$), and absolute percentage errors for the benchmark Log-Linear model.


In [4]:
print("=== TOP 10 LARGEST ABSOLUTE RESIDUAL ERRORS (HELD-OUT 2022) ===")
display(err_df.head(10))

print("\n=== TOP 5 LOWEST ABSOLUTE RESIDUAL ERRORS (HELD-OUT 2022) ===")
display(err_df.tail(5))


=== TOP 10 LARGEST ABSOLUTE RESIDUAL ERRORS (HELD-OUT 2022) ===

=== TOP 5 LOWEST ABSOLUTE RESIDUAL ERRORS (HELD-OUT 2022) ===


,state_name,target_year,actual_volume,predicted_volume,signed_error,absolute_error,abs_percentage_error
0,Telangana,2022,15297.0,10612.19,4684.81,4684.81,30.63
1,Assam,2022,1733.0,5730.48,-3997.48,3997.48,230.67
2,Uttar Pradesh,2022,10117.0,12472.17,-2355.17,2355.17,23.28
3,Maharashtra,2022,8249.0,7273.17,975.83,975.83,11.83
4,Karnataka,2022,12556.0,11688.40,867.60,867.60,6.91
5,Tamil Nadu,2022,2082.0,1283.27,798.73,798.73,38.36
6,Odisha,2022,1983.0,2643.32,-660.32,660.32,33.30
7,Gujarat,2022,1417.0,1914.39,-497.39,497.39,35.10
8,Jharkhand,2022,967.0,1367.06,-400.06,400.06,41.37
9,West Bengal,2022,401.0,762.26,-361.26,361.26,90.09


,state_name,target_year,actual_volume,predicted_volume,signed_error,absolute_error,abs_percentage_error
31,Chandigarh,2022,27.0,21.63,5.37,5.37,19.89
32,Madhya Pradesh,2022,826.0,830.04,-4.04,4.04,0.49
33,Ladakh,2022,3.0,4.70,-1.70,1.70,56.67
34,Lakshadweep,2022,1.0,2.47,-1.47,1.47,147.00
35,Dadra and Nagar Haveli and Daman and Diu,2022,5.0,6.17,-1.17,1.17,23.40


## 5. Residual Diagnostics & Skewness Analysis

We examine residual error distributions and skewness across all candidate architectures.


In [5]:
print("=== RESIDUAL ERROR SUMMARY & SKEWNESS ACROSS ARCHITECTURES ===")
display(res_summary_df)


=== RESIDUAL ERROR SUMMARY & SKEWNESS ACROSS ARCHITECTURES ===


,Model,Mean_Residual,Std_Residual,Min_Residual,Max_Residual,Residual_Skewness
0,Naive Persistent (Lag-1),358.86,1291.84,-3113.00,4994.00,1.8448
1,Historical 2-Year Moving Average,399.68,1470.37,-2455.00,7633.50,3.5269
2,Linear Regression (OLS Raw),371.62,1638.51,-3050.00,6829.48,2.2971
3,"Ridge Regression (Raw, alpha=1.0)",371.62,1638.51,-3050.00,6829.48,2.2971
4,Log-Linear OLS (Stage 7 Benchmark),-41.51,1142.71,-3997.48,4684.81,0.6472
5,Polynomial Degree 2 (OLS Raw),429.83,1750.98,-4364.56,7292.88,1.7450
6,"Polynomial Degree 2 (Ridge, alpha=100.0)",429.81,1750.95,-4364.56,7292.70,1.7450
7,Log-Polynomial Degree 2 (OLS),331.07,1484.04,-3235.31,5668.57,1.9981
8,"Log-Polynomial Degree 2 (Ridge, alpha=1.0)",308.10,1433.07,-3295.48,5358.58,1.9046
9,Decision Tree Regressor (depth=3),328.30,1536.80,-3763.00,7089.50,2.4611


## 6. Visualizations & Analytical Figures

We generate and render the Stage 14 visualization suite (Figures 45 to 49).


In [6]:
# Generate and save figures and tables
test_mask = panel_df['split'] == 'test'
y_test_raw = panel_df.loc[test_mask, 'target_actual'].values

plot_regression_enhancement_figures(
    comp_df, test_pred_df, err_df, models_preds, y_test_raw
)

model_sel_df = pd.DataFrame([
    {
        'Stage': 'Stage 7 (Baseline)',
        'Selected_Model': 'Log-Linear OLS',
        'Model_Complexity': 'Low (2 Linear Log Parameters)',
        'Test_MAE': 479.37,
        'Test_R2': 0.9000,
        'Rationale': 'Log transform stabilizes multi-order variance across states without parameter explosion.'
    },
    {
        'Stage': 'Stage 14 (Enhanced)',
        'Selected_Model': 'Log-Linear OLS (Retained Benchmark)',
        'Model_Complexity': 'Low (2 Linear Log Parameters)',
        'Test_MAE': 479.37,
        'Test_R2': 0.9000,
        'Rationale': 'Non-linear polynomial/tree/boosting extensions increase out-of-sample error on N=70 sample.'
    }
])
save_stage14_tables(comp_df, test_pred_df, err_df, res_summary_df, model_sel_df)
print("Visualizations (Figures 45-49) and Tables successfully generated.")


Visualizations (Figures 45-49) and Tables successfully generated.


## 7. Model Complexity vs. Performance (Occam's Razor in Small-$N$ Panels)

### Empirical Finding:
> **"Does nonlinear modeling provide meaningful predictive improvement over the already strong Log-Linear baseline?"**

**Result**: **No.** Adding non-linear polynomial expansions (Degree-2 and Degree-3), regression trees, or ensemble boosting did not improve out-of-sample predictive accuracy over the 2-parameter Log-Linear OLS model ($R^2 = 0.9000, \text{MAE} = 479.37$).

### Methodological Explanation:
1. **Variance Stabilization**: Indian state cybercrime volumes span 4 orders of magnitude ($N=1$ in Lakshadweep/Ladakh to $N=20,000+$ in Karnataka/Telangana). The logarithmic transformation $\log(1+y)$ stabilizes this extreme heteroscedastic scale variance.
2. **Overfitting on Small Sample ($N_{\text{train}} = 70$)**: Polynomial expansions create interaction terms ($y_{t-1}^2, y_{t-2}^2, y_{t-1}y_{t-2}$) that overfit the volatile training years (2020 pandemic surge) and inflate test error on 2022.
3. **Step-Function Inefficiency**: Decision Trees and Random Forests partition feature space into piecewise constant step functions, which cannot smoothly extrapolate continuous volume growth as effectively as a log-linear trajectory.

---

## 8. Comparison: Stage 13 (Classification) vs. Stage 14 (Regression)

| Analytical Dimension | Stage 13 — Classification | Stage 14 — Regression |
| :--- | :--- | :--- |
| **Analytical Question** | "Will next-year volume belong to a high-volume regime?" | "What will the exact numerical volume be in year $t$?" |
| **Target Type ($Y$)** | Binary Discrete: $Y_t \in \{0, 1\}$ ($\ge 367.0$ cases) | Continuous Magnitude: $\hat{Y}_t \in [0, \infty)$ cases |
| **Evaluation Metrics** | Accuracy, Precision, Recall, Specificity, F1, ROC-AUC | MAE, RMSE, $R^2$, Median Absolute Error |
| **Best Model** | Linear SVM / Random Forest / Naive Bayes ($\text{Acc} = 1.0, F_1 = 1.0$) | Log-Linear OLS ($R^2 = 0.9000, \text{MAE} = 479.4$) |
| **Information Preserved** | Binary regime category (high vs. low) | Full numerical magnitude, scale, and rate of growth |
| **Operational Utility** | Macro administrative tiering & resource threshold triaging | Budgeting, infrastructure capacity, and personnel allocation |

---

## 9. Limitations & Academic Conclusions

1. **Small-$N$ Panel Constraints**: The test horizon contains exactly $N = 36$ State/UT jurisdictions across 5 historical years ($N=106$ total observations).
2. **Volatile Regional Reporting Changes**: High absolute errors occur in states experiencing dramatic administrative reporting shifts (e.g., Assam dropping from 4,846 in 2021 to 1,733 in 2022; Telangana surging from 10,303 in 2021 to 15,297 in 2022).
3. **Non-Causal Interpretation**: Models forecast aggregate statistical volume inertia; they do not isolate criminal intent, enforcement effectiveness, or socio-demographic causation.
4. **Syllabus Educational Purpose**: This stage demonstrates the principle of **Occam's Razor** in supervised learning: simpler transformed linear models frequently outperform complex non-linear ensembles on small, heavy-tailed macroeconomic panels.
